In [1]:
import re
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import clone_model
from tensorflow.keras.applications import EfficientNetV2B0
from sklearn.metrics import accuracy_score, classification_report
from transformers import AutoModel, TFAutoModel, AutoTokenizer, BertTokenizer
from PIL import Image
import pandas as pd
from sklearn.model_selection import train_test_split
from bs4 import BeautifulSoup
import requests
from io import BytesIO
import cv2
import matplotlib.pyplot as plt

2025-09-28 00:46:21.707067: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
class BertEmbeddingLayer(tf.keras.layers.Layer):
    def __init__(self, model_name, **kwargs):
        super(BertEmbeddingLayer, self).__init__(**kwargs)
        self.model_name = model_name
        self.embedding_size = None
        self.bert = None

    def build(self, input_shape):
        self.bert = TFAutoModel.from_pretrained(self.model_name)
        self.embedding_size = self.bert.config.hidden_size
        super(BertEmbeddingLayer, self).build(input_shape)

    def call(self, inputs):
        ids, att = inputs
        outputs = self.bert(input_ids=ids, attention_mask=att)
        return outputs.pooler_output

    def get_config(self):
        config = super(BertEmbeddingLayer, self).get_config()
        config.update({
            "model_name": self.model_name
        })
        return config
    
    @classmethod
    def from_config(cls, config):
        clean_config = {k: v for k, v in config.items() 
                      if k in ['model_name', 'name', 'trainable', 'dtype']}
        return cls(**clean_config)

tf.keras.utils.get_custom_objects()['BertEmbeddingLayer'] = BertEmbeddingLayer

In [3]:
def makemodel(output_bias=None):
    # Text using IndoBertp1
    ids1 = tf.keras.layers.Input(shape=(32,), dtype=tf.int32, name="title")
    att1 = tf.keras.layers.Input(shape=(32,), dtype=tf.int32, name="titlemask")
    ids2 = tf.keras.layers.Input(shape=(128,), dtype=tf.int32, name="content")
    att2 = tf.keras.layers.Input(shape=(128,), dtype=tf.int32, name="contentmask")

    indobert1 = BertEmbeddingLayer(model_name="indobenchmark/indobert-base-p1", name="indobert1")
    indobert2 = BertEmbeddingLayer(model_name="indobenchmark/indobert-base-p1", name="indobert2")

    title = indobert1([ids1, att1])
    content = indobert2([ids2, att2])

    # Image using EfficientNetV2s
    img1 = tf.keras.layers.Input(shape=(224,224,3), dtype=tf.uint8, name="image")

    img_aug = tf.keras.layers.RandomRotation(0.2)(img1)
    img_aug = tf.keras.layers.RandomTranslation(0.1, 0.1)(img_aug)
    img_aug = tf.keras.layers.RandomFlip('horizontal')(img_aug)
    img_aug = tf.keras.layers.RandomContrast(0.2)(img_aug)

    effi = EfficientNetV2B0(
        include_top = False,
        weights = 'imagenet',
        input_shape = (224,224,3),
        pooling = 'max'
    )
    image = effi(img_aug)

    fin = tf.keras.layers.concatenate([title, content, image])
    fin = tf.keras.layers.BatchNormalization()(fin)
        
    fin = tf.keras.layers.Dense(512, activation=None)(fin)
    fin = tf.keras.layers.BatchNormalization()(fin)
    fin = tf.keras.layers.Activation('relu')(fin)
    fin = tf.keras.layers.Dropout(0.4)(fin) 

    fin = tf.keras.layers.Dense(256, activation=None)(fin)
    fin = tf.keras.layers.BatchNormalization()(fin)
    fin = tf.keras.layers.Activation('relu')(fin)
    fin = tf.keras.layers.Dropout(0.4)(fin) 
    
    fin = tf.keras.layers.Dense(128, activation=None)(fin)
    fin = tf.keras.layers.BatchNormalization()(fin)
    fin = tf.keras.layers.Activation('relu')(fin)
    fin = tf.keras.layers.Dropout(0.4)(fin) 
    
    fin = tf.keras.layers.Dense(64, activation='relu')(fin)
    fin = tf.keras.layers.Dropout(0.2)(fin)
    fin = tf.keras.layers.Dense(1, activation='sigmoid')(fin)

    final = tf.keras.Model(inputs=[ids1, att1, ids2, att2, img1], outputs=fin)

    for layer in final.layers:
      layer.trainable = True
    optimizer = tf.keras.optimizers.Adam(learning_rate = 0.00005)
    final.compile(
        loss = tf.keras.losses.BinaryCrossentropy(),
        optimizer=optimizer,
        metrics=[tf.keras.metrics.BinaryAccuracy(),tf.keras.metrics.Precision(),tf.keras.metrics.Recall()]
    )

    return final

final = makemodel()

2025-09-28 00:46:27.087965: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:05:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-09-28 00:46:27.103311: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:05:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-09-28 00:46:27.103363: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:05:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-09-28 00:46:27.105682: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:05:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-09-28 00:46:27.105732: I external/local_xla/xla/stream_executor

In [4]:
dataset = tf.data.Dataset.load(
    "dataset",
    element_spec=({
        'title': tf.TensorSpec(shape=(None, 32), dtype=tf.int32),
        'content': tf.TensorSpec(shape=(None, 128), dtype=tf.int32),
        'titlemask': tf.TensorSpec(shape=(None, 32), dtype=tf.int32),
        'contentmask': tf.TensorSpec(shape=(None, 128), dtype=tf.int32),
        'image': tf.TensorSpec(shape=(None, 224, 224, 3), dtype=tf.uint8)
    }, tf.TensorSpec(shape=(None,), dtype=tf.int32))
)

dataVal = tf.data.Dataset.load(
    "dataVal",
    element_spec=({
        'title': tf.TensorSpec(shape=(None, 32), dtype=tf.int32),
        'content': tf.TensorSpec(shape=(None, 128), dtype=tf.int32),
        'titlemask': tf.TensorSpec(shape=(None, 32), dtype=tf.int32),
        'contentmask': tf.TensorSpec(shape=(None, 128), dtype=tf.int32),
        'image': tf.TensorSpec(shape=(None, 224, 224, 3), dtype=tf.uint8)
    }, tf.TensorSpec(shape=(None,), dtype=tf.int32))
)

In [5]:
tf.keras.mixed_precision.set_global_policy('mixed_float16')

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
    )

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=1,
    min_lr=1e-8
)

history=final.fit(x = dataset,
                  epochs = 20,
                  callbacks=[early_stopping, 
                             reduce_lr],
                  validation_data=dataVal
                  )

Epoch 1/20


2025-09-28 00:47:30.752978: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/functional_1/efficientnetv2-b0_1/block2b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
2025-09-28 00:47:36.004336: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:465] Loaded cuDNN version 8907


350/350 ━━━━━━━━━━━━━━━━━━━━ 254s 581ms/step - binary_accuracy: 0.5572 - loss: 0.7206 - precision: 0.4858 - recall: 0.3618 - val_binary_accuracy: 0.6249 - val_loss: 0.6586 - val_precision: 0.6436 - val_recall: 0.2658 - learning_rate: 5.0000e-05
Epoch 2/20
350/350 ━━━━━━━━━━━━━━━━━━━━ 196s 560ms/step - binary_accuracy: 0.5976 - loss: 0.6819 - precision: 0.5446 - recall: 0.4366 - val_binary_accuracy: 0.6815 - val_loss: 0.6120 - val_precision: 0.6685 - val_recall: 0.4987 - learning_rate: 5.0000e-05
Epoch 3/20
350/350 ━━━━━━━━━━━━━━━━━━━━ 196s 561ms/step - binary_accuracy: 0.6293 - loss: 0.6548 - precision: 0.5867 - recall: 0.4895 - val_binary_accuracy: 0.6951 - val_loss: 0.5864 - val_precision: 0.6753 - val_recall: 0.5458 - learning_rate: 5.0000e-05
Epoch 4/20
350/350 ━━━━━━━━━━━━━━━━━━━━ 196s 560ms/step - binary_accuracy: 0.6589 - loss: 0.6253 - precision: 0.6303 - recall: 0.5150 - val_binary_accuracy: 0.7044 - val_loss: 0.5757 - val_precision: 0.6820 - val_recall: 0.5719 - learning_rate

In [12]:
tokenizer = BertTokenizer.from_pretrained("indobenchmark/indobert-base-p1")

def preprocess_text(text, max_length):
    encoded = tokenizer.encode_plus(
    text,
    max_length = max_length,
    padding = 'max_length',
    truncation = True,
    return_tensors='tf'
    )
    return encoded['input_ids'], encoded['attention_mask']

def preprocess_image(img):
    img = tf.image.decode_image(img, channels=3)
    img = tf.image.resize_with_pad(img, 224, 224)
    img = tf.cast(img,tf.uint8)
    img = tf.expand_dims(img, axis=0)

    # plt.figure(figsize=(8, 8))
    # plt.imshow(img[0])
    # plt.axis('off')
    # plt.show()

    return img

def predict_single(model, title_text, content_text, image_input):
    title_ids, title_mask = preprocess_text(title_text, max_length=32)
    content_ids, content_mask = preprocess_text(content_text, max_length=128)

    image = preprocess_image(image_input)

    # if image.dtype == tf.float32 and tf.reduce_max(image) <= 1.0:
    #     image_processed = tf.cast(image * 255.0, tf.uint8)
    # else:
    #     image_processed = tf.cast(image, tf.uint8)
    
    # prediction = model.serve({
    #     'title': tf.cast(title_ids, tf.int32),
    #     'titlemask': tf.cast(title_mask, tf.int32),
    #     'content': tf.cast(content_ids, tf.int32),
    #     'contentmask': tf.cast(content_mask, tf.int32),
    #     'image': image_processed
    # })

    prediction = model.predict({
        'title': title_ids,
        'titlemask': title_mask,
        'content': content_ids,
        'contentmask': content_mask,
        'image': image
    })

    # prediction = model.serve([
    #     title_ids,     
    #     title_mask,    
    #     content_ids,  
    #     content_mask,  
    #     image         
    # ])

    return prediction[0][0]

def predict_single_detik(model,link):
    website = requests.get(link)
    html = website.text
    soup = BeautifulSoup(html)
    img = soup.find('meta', {'name':'thumbnailUrl'})['content']
    title = soup.find('h1',{'class':"detail__title"}).get_text(strip=True)
    content = [p.get_text() for p in soup.find_all('p')]
    content = ''.join(s for s in content if s.strip() and '[' not in s and 'SCROLL' not in s and 'Lihat Video' not in s)

    response = requests.get(img)

    print("title: ",title)
    print("content: ",content[:200])

    return predict_single(model,title,content,response.content)


In [14]:
testdf = pd.read_csv("nissan extended.csv", sep=';')
testdf.head(1)
testdf['result'] = None
for index, row in testdf.iterrows():
    title = row['title']
    content = row['content']
    img = tf.io.read_file(row['image_path_processed'])
    testdf.loc[index,'result'] = predict_single(final,title,content,img)
    testdf.loc[index,'diff'] = testdf['result'][index]-testdf['result'][0]
    if index%10 == 0:
        print(index, 'done')
    

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step
0 done
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 130ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 129ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 129ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 198ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 129ms/step
10 done
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step
20 done
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 129ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step
1/

In [15]:
testdf.head(5)

,Unnamed: 0,title,content,image_path_processed,label_score,Judul,Isi,Gambar,result,diff
0,0,Nissan Kasih Garansi Baterai Leaf Sampai 8 Tahun,"- PT , Motor Indonesia (NMI) telah memastika...",click-id/processed_images/kompas/1236.jpg,0,NaN,NaN,NaN,0.26568,0.000000
1,1,Nissan Kasih Garansi Baterai Leaf Sampai 8 Tahun,"- PT , Motor Indonesia (NMI) telah memastika...",black.jpg,0,NaN,NaN,gambar hitam,0.351159,0.085478
2,2,Nissan Kasih Garansi Baterai Leaf Sampai 8 Tahun,"- PT , Motor Indonesia (NMI) telah memastika...",contrast.jpg,0,NaN,NaN,gambar kontras,0.24723,-0.018450
3,3,Nissan Kasih Garansi Baterai Leaf Sampai 8 Tahun,"- PT , Motor Indonesia (NMI) telah memastika...",bright.jpg,0,NaN,NaN,gambar terang,0.239941,-0.025739
4,4,Nissan Kasih Garansi Baterai Leaf Sampai 8 Tahun,"- PT , Motor Indonesia (NMI) telah memastika...",illust.jpg,0,NaN,NaN,gambar ilustrasi,0.226565,-0.039115


In [16]:
testdf.to_csv("result extended.csv")

In [17]:
testdf2 = pd.read_csv("testing nissan - demo.csv")
testdf2.head(1)
testdf2['result'] = None
for index, row in testdf2.iterrows():
    title = row['title']
    content = row['content']
    img = tf.io.read_file(row['image_path_processed'])
    testdf2.loc[index,'result'] = predict_single(final,title,content,img)
    testdf2.loc[index,'diff'] = testdf['result'][index]-testdf['result'][0]
    if index%10 == 0:
        print(index, 'done')
    

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 190ms/step
0 done
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 196ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 129ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 129ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 130ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 129ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 206ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 145ms/step
10 done
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 131ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 131ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step
20 done
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 150ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step
1/1

In [18]:
testdf2.head(5)

,title,content,image_path_processed,label_score,explanation,result,diff
0,Nissan Kasih Garansi Baterai Leaf Sampai 8 Tahun,"- PT , Motor Indonesia (NMI) telah memastika...",click-id/processed_images/kompas/1236.jpg,0,original,0.26568,0.000000
1,Heboh!!! Nissan Kasih Garansi Baterai Leaf Sam...,"- PT , Motor Indonesia (NMI) telah memastika...",click-id/processed_images/kompas/1236.jpg,0,dramatis,0.577883,0.085478
2,"Geger, Nissan Kasih Garansi Baterai Leaf Sampa...","- PT , Motor Indonesia (NMI) telah memastika...",click-id/processed_images/kompas/1236.jpg,0,dramatis,0.321021,-0.018450
3,Mengerikan? Nissan Kasih Garansi Baterai Leaf ...,"- PT , Motor Indonesia (NMI) telah memastika...",click-id/processed_images/kompas/1236.jpg,0,dramatis,0.407011,-0.025739
4,WOW! Nissan Kasih Garansi Baterai Leaf Sampai ...,"- PT , Motor Indonesia (NMI) telah memastika...",click-id/processed_images/kompas/1236.jpg,0,dramatis,0.353149,-0.039115


In [19]:
testdf2.to_csv("result base.csv")